In [11]:
import sys, json
from pathlib import Path

#  project root on path 
ROOT = Path("../").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")

#  artifact paths 
FAISS_MANIFEST = ROOT / "artifacts/faiss/chunks_embedding.Qwen__Qwen3-Embedding-0.6B.nlp_prefix.FlatIP.manifest.json"
META_JSONL     = ROOT / "artifacts/faiss/chunks_embedding.Qwen__Qwen3-Embedding-0.6B.nlp_prefix.FlatIP.meta.jsonl"
BENCHMARK      = ROOT / "benchmark/benchmark.json"

#  models 
# HuggingFace ID or local path; pipeline resolves model_checkpoints/ automatically
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

#  retrieval hyper-parameters 
BM25_TOP_K      = 100   # BM25 candidates per query
DENSE_TOP_K     = 100   # Dense candidates per query
RRF_K           = 60    # RRF smoothing constant
RRF_WEIGHTS     = [1.0, 1.0]   # [bm25_weight, dense_weight]
RERANK_INPUT_K  = 30    # candidates fed to cross-encoder
FINAL_TOP_K     = 5     # results returned per query

#  evaluation 
EVAL_KS         = [3, 5]   # cut-offs for Recall@k and NDCG@k
EVAL_VERBOSE    = False           # print per-question hit/miss

Project root: /home/arkkuma/final


## Load Pipeline

In [12]:
from pipeline import RetrievalPipeline

pipeline = RetrievalPipeline(
    meta_jsonl_path     = META_JSONL,
    faiss_manifest_path = FAISS_MANIFEST,
    cross_encoder_model = CROSS_ENCODER_MODEL,
    bm25_top_k          = BM25_TOP_K,
    dense_top_k         = DENSE_TOP_K,
    rrf_k               = RRF_K,
    rrf_weights         = RRF_WEIGHTS,
    rerank_input_k      = RERANK_INPUT_K,
    final_top_k         = FINAL_TOP_K,
)
print("Pipeline ready.")

[BM25] Indexed 1404 docs from chunks_embedding.Qwen__Qwen3-Embedding-0.6B.nlp_prefix.FlatIP.meta.jsonl
[Dense] FAISS loaded: 1404 vectors, dim=1024 from chunks_embedding.Qwen__Qwen3-Embedding-0.6B.nlp_prefix.FlatIP.index
[Reranker] Using HuggingFace: cross-encoder/ms-marco-MiniLM-L-6-v2
[Reranker] Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline ready.


## Evaluate on Benchmark

Ground-truth matching: title + section (strips `(part N)` suffixes).

In [13]:
from evaluation.metrics import load_benchmark, evaluate, print_metrics

benchmark = load_benchmark(BENCHMARK)
print(f"Benchmark: {len(benchmark)} questions")

metrics = evaluate(
    pipeline,
    benchmark,
    ks=EVAL_KS,
    verbose=EVAL_VERBOSE,
)

print_metrics(metrics)

Benchmark: 50 questions
[Dense] Local model not found, using HuggingFace: Qwen/Qwen3-Embedding-0.6B
[Dense] Loading query encoder: Qwen/Qwen3-Embedding-0.6B


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


  Metric              Score
----------------------------------------
  MRR                0.8057
  Recall@3           0.8800
  NDCG@3             0.8109
  Recall@5           0.9600
  NDCG@5             0.8445


## BM25-only vs Dense-only vs BM25+Dense+Rerank

In [14]:
from evaluation.metrics import recall_at_k, reciprocal_rank, ndcg_at_k

ABLATION_K = 5   # cut-off for ablation comparison

def _eval_single_stage(retrieve_fn, benchmark, k=ABLATION_K):
    """Evaluate a single-stage retriever (BM25-only or Dense-only)."""
    rr_list, rec_list, ndcg_list = [], [], []
    for item in benchmark:
        # Single-stage retrievers return raw meta dicts that contain vector_id
        metas = retrieve_fn(item["query"])
        gt_id = item["gt_chunk_id"]
        rr_list.append(reciprocal_rank(metas, gt_id))
        rec_list.append(recall_at_k(metas, gt_id, k))
        ndcg_list.append(ndcg_at_k(metas, gt_id, k))
    n = len(benchmark)
    return {
        "MRR":          sum(rr_list)  / n,
        f"Recall@{k}":  sum(rec_list) / n,
        f"NDCG@{k}":    sum(ndcg_list)/ n,
    }

def _bm25_retrieve(query):
    hits = pipeline.bm25.search(query, top_k=max(EVAL_KS))
    return [meta for _, _, meta in hits]

def _dense_retrieve(query):
    hits = pipeline.dense.search(query, top_k=max(EVAL_KS))
    return [meta for _, _, meta in hits]

print("Evaluating BM25-only ...")
m_bm25  = _eval_single_stage(_bm25_retrieve, benchmark)

print("Evaluating Dense-only ...")
m_dense = _eval_single_stage(_dense_retrieve, benchmark)

print(f"\n Ablation @ k={ABLATION_K} ")
print(f"{'Metric':<15} {'BM25':>8} {'Dense':>8} {'Full Pipeline':>14}")
print("-" * 50)
for key in m_bm25:
    print(f"{key:<15} {m_bm25[key]:>8.4f} {m_dense[key]:>8.4f} {metrics.get(key, 0):>14.4f}")

Evaluating BM25-only ...
Evaluating Dense-only ...

 Ablation @ k=5 
Metric              BM25    Dense  Full Pipeline
--------------------------------------------------
MRR               0.6417   0.8007         0.8057
Recall@5          0.7800   0.9400         0.9600
NDCG@5            0.6769   0.8359         0.8445
